## **Análisis Cuantitativo del Impacto de los Flujos de Capital hacia ETFs en la Valoración del S&P500**


Este estudio tiene como objetivo realizar una contribución significativa a la investigación financiera existente. Mediante la expansión del modelo Fama-French, busca explorar la influencia de los flujos de fondos —específicamente, el capital que entra y sale de los fondos de inversión que tienen participaciones en Apple— en los rendimientos de las acciones de la empresa.

El modelo ampliado se formula de la siguiente manera:


R_{apple,t} - R_{f,t} = \alpha + \beta_m(R_{m,t} - R_{f,t}) + \beta_sSMB_t + \beta_hHML_t + \beta_fFF_t + \epsilon_t

Donde:
- `R_{apple,t} - R_{f,t}` es el exceso de rendimiento de las acciones de Apple sobre la tasa libre de riesgo en el tiempo t.
- `\alpha` es el término de intercepción del modelo.
- `\beta_m, \beta_s, \beta_h, \beta_f` son los coeficientes de sensibilidad a los factores de mercado, tamaño, valor, y flujos de fondos, respectivamente.
- `R_{m,t} - R_{f,t}` representa el exceso de rendimiento del mercado.
- `SMB_t` y `HML_t` son los factores de tamaño y valor del modelo Fama-French.
- `FF_t` es el factor adicional de flujos de fondos propuesto.
- `\epsilon_t` es el término de error.



Este enfoque permite evaluar si los flujos monetarios hacia o desde fondos de inversión ejercen una influencia significativa en la valoración de Apple, aportando una nueva dimensión al análisis tradicional de rendimientos de acciones.

In [63]:
import pandas as pd
import statsmodels.api as sm

# Cargar el CSV en un DataFrame
df = pd.read_csv('/content/general_csv_weekly_4factors_finance.csv', sep = ';')

print(df)

           Date Mkt-RF   SMB    HML     RF Fundflows   rent_sector
0    02/02/2018  -3,81  -0,5  -0,11  0,029     2,476  -0,027763644
1    09/02/2018  -5,09  1,05   0,09  0,029    -6,931  -0,054505789
2    16/02/2018   4,51  0,09  -0,54  0,029     1,688   0,048446431
3    23/02/2018   0,48  0,12  -0,59  0,029    -0,323   0,005640056
4    02/03/2018  -1,74  1,02  -0,86  0,029     0,855  -0,021506426
..          ...    ...   ...    ...    ...       ...           ...
303  24/11/2023   0,91     0  -0,76   0,11    -0,395   0,010670561
304  01/12/2023   1,01  1,63   1,75  0,107     1,515   0,026801287
305  08/12/2023   0,21  0,95   0,98  0,107     1,093   0,000167378
306  15/12/2023   2,72  1,62   1,83  0,107     1,679   0,041273951
307  22/12/2023   0,86  1,97   0,52  0,107     3,655   0,001173479

[308 rows x 7 columns]


In [55]:
df['Date'] = pd.to_datetime(df['Date'])
print(df.columns)
print(df)
print(df.dtypes)
format='%y/%m/%d'

ValueError: time data "16/02/2018" doesn't match format "%m/%d/%Y", at position 2. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [64]:
# Eliminar la última fila utilizando .drop() y iloc
df.drop(df.iloc[-1].name, inplace=True)



# Verifica tu DataFrame
print(df)

# Verifica los cambios
print(df.head())
print(df.dtypes)


           Date Mkt-RF   SMB    HML     RF Fundflows   rent_sector
0    02/02/2018  -3,81  -0,5  -0,11  0,029     2,476  -0,027763644
1    09/02/2018  -5,09  1,05   0,09  0,029    -6,931  -0,054505789
2    16/02/2018   4,51  0,09  -0,54  0,029     1,688   0,048446431
3    23/02/2018   0,48  0,12  -0,59  0,029    -0,323   0,005640056
4    02/03/2018  -1,74  1,02  -0,86  0,029     0,855  -0,021506426
..          ...    ...   ...    ...    ...       ...           ...
302  17/11/2023   2,46  2,67   1,73   0,11     1,423   0,034448372
303  24/11/2023   0,91     0  -0,76   0,11    -0,395   0,010670561
304  01/12/2023   1,01  1,63   1,75  0,107     1,515   0,026801287
305  08/12/2023   0,21  0,95   0,98  0,107     1,093   0,000167378
306  15/12/2023   2,72  1,62   1,83  0,107     1,679   0,041273951

[307 rows x 7 columns]
         Date Mkt-RF   SMB    HML     RF Fundflows   rent_sector
0  02/02/2018  -3,81  -0,5  -0,11  0,029     2,476  -0,027763644
1  09/02/2018  -5,09  1,05   0,09  0,029  

In [65]:
import pandas as pd

# Asegúrate de que todas las columnas sean de tipo string antes de reemplazar
df['Mkt-RF'] = df['Mkt-RF'].astype(str).str.replace(',', '.').astype(float) / 100
df['SMB'] = df['SMB'].astype(str).str.replace(',', '.').astype(float) / 100
df['HML'] = df['HML'].astype(str).str.replace(',', '.').astype(float) / 100

# Convierte las columnas de RF y rendimientos de apple de porcentaje a float
df['RF'] = df['RF'].astype(str).str.replace(',', '.').astype(float) / 100
df['rent_sector'] = df['rent_sector'].astype(str).str.replace(',', '.').astype(float)

# Para % Fundflows, maneja los NaNs antes de la conversión si es necesario
df['Fundflows'] = df['Fundflows'].astype(str).str.replace(',', '.').astype(float) / 100

# Verifica los cambios
print(df.head())
print(df.dtypes)

         Date  Mkt-RF     SMB     HML       RF  Fundflows  rent_sector
0  02/02/2018 -0.0381 -0.0050 -0.0011  0.00029    0.02476    -0.027764
1  09/02/2018 -0.0509  0.0105  0.0009  0.00029   -0.06931    -0.054506
2  16/02/2018  0.0451  0.0009 -0.0054  0.00029    0.01688     0.048446
3  23/02/2018  0.0048  0.0012 -0.0059  0.00029   -0.00323     0.005640
4  02/03/2018 -0.0174  0.0102 -0.0086  0.00029    0.00855    -0.021506
Date            object
Mkt-RF         float64
SMB            float64
HML            float64
RF             float64
Fundflows      float64
rent_sector    float64
dtype: object


In [69]:
import statsmodels.api as sm

# Preparar las variables independientes
# Añadir una constante al modelo para el término de intercepción
X = df[['Mkt-RF', 'SMB', 'HML', 'Fundflows']]
X = sm.add_constant(X)

# La variable dependiente es el exceso de rendimiento de Apple
# Asegúrate de que 'RF' y 'rendimientos apple' están en formato decimal adecuado
Y = df['rent_sector'] - df['RF']

# Estimar el modelo OLS
modelo = sm.OLS(Y, X).fit()

# Mostrar el resumen del modelo
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.921
Model:                            OLS   Adj. R-squared:                  0.920
Method:                 Least Squares   F-statistic:                     882.7
Date:                Sat, 18 May 2024   Prob (F-statistic):          3.30e-165
Time:                        14:22:29   Log-Likelihood:                 985.47
No. Observations:                 307   AIC:                            -1961.
Df Residuals:                     302   BIC:                            -1942.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0007      0.001     -1.165      0.2

In [68]:
import pandas as pd
import statsmodels.api as sm



# Step 2: Filter the DataFrame for the desired date range
start_date = '2020-01-01'
end_date = '2023-12-31'
mask = (df['Date'] >= start_date) & (df['Date'] <= end_date)
filtered_df = df.loc[mask]

# Step 3: Prepare the independent variables, add a constant for the intercept
X = filtered_df[['Mkt-RF', 'SMB', 'HML', 'Fundflows']]
X = sm.add_constant(X)

# Ensure 'RF' and 'rendimientos tesla' are in the correct decimal format
Y = filtered_df['rent_sector'] - filtered_df['RF']

# Estimate the OLS model
modelo = sm.OLS(Y, X).fit()

# Show the model summary
print(modelo.summary())

ValueError: zero-size array to reduction operation maximum which has no identity